# Extractor Node — Unit Tests
Step-by-step testing of the Extractor Node functions.
Run each cell independently to inspect intermediate outputs and logs.

**Pre-requisites before running:**
1. `OPENAI_API_KEY` set in `.env`
2. Qdrant running with the `openapi_reference` collection indexed (run `test_rag.ipynb` first)

In [ ]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE
from nodes.reader import reader_node
from nodes.planner import planner_node
from nodes.extractor import extractor_node, _build_sections_index, _get_sections_to_reprocess, _build_correction_task, _build_allowed_keys
from schemas.rules import RawRule

logger = get_logger(__name__)
logger.info("Imports OK.")

In [2]:
# Step 2 — Run Reader + Planner to prepare state
# The Extractor depends on both Reader and Planner output.
# This replicates the state that LangGraph would pass into extractor_node().

reader_result = reader_node({
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})

assert len(reader_result["parsed_sections"]) > 0, "Reader returned no sections"
logger.info(f"Reader OK — {len(reader_result['parsed_sections'])} section(s).")

planner_result = planner_node({
    **reader_result,
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})

assert "extraction_plan" in planner_result
n_planned = len(planner_result["extraction_plan"]["sections_to_extract"])
logger.info(f"Planner OK — {n_planned} section(s) selected for extraction.")

2026-03-30 21:37:44 [INFO] nodes.reader: Reader Node started.
2026-03-30 21:37:44 [DEBUG] nodes.reader: Parsed 238 relevant sections from main doc.
2026-03-30 21:37:44 [DEBUG] nodes.reader: Loaded 0 auxiliary document(s).
2026-03-30 21:37:44 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-30 21:37:44 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-30 21:37:44 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_reference'.
2026-03-30 21:37:44 [INFO] nodes.reader: Reader Node complete.
2026-03-30 21:37:44 [INFO] __main__: Reader OK — 238 section(s).
2026-03-30 21:37:44 [INFO] nodes.planner: Planner Node started.
2026-03-30 21:37:44 [DEBUG] nodes.planner: Sending 238 section(s) to Planner LLM.
2026-03-30 21:37:44 [DEBUG] open

In [3]:
# Step 3 — Test _build_sections_index()
# Verifies the lookup dict is built correctly from parsed_sections.

index = _build_sections_index(reader_result["parsed_sections"])

assert isinstance(index, dict)
assert len(index) == len(reader_result["parsed_sections"])
assert all("section_id" in v and "title" in v and "content" in v for v in index.values())

sample_id = reader_result["parsed_sections"][0]["section_id"]
assert sample_id in index

logger.info(f"_build_sections_index OK — {len(index)} entries.")
logger.info(f"  Sample lookup [{sample_id}]: {index[sample_id]['title']}")

2026-03-30 21:38:20 [INFO] __main__: _build_sections_index OK — 238 entries.
2026-03-30 21:38:20 [INFO] __main__:   Sample lookup [1]: 11.1.0 Introduction


In [ ]:
# Step 4 — Test _get_sections_to_reprocess(), _build_correction_task(), _build_allowed_keys()
# Verifies that on loop-back only failing sections are selected,
# the correction prompt is correctly formatted, and allowed keys are built correctly.

mock_errors = [
    {
        "rule": {
            "section_id":   "322",
            "rule_type":    "path_operation",
            "source_name":  "modifyMOIAttributes",
            "rule_text":    "modifyMOIAttributes maps to PUT and PATCH",
            "openapi_mapping": {
                "openapi_object": "paths./ProvMnS/{MnSVersion}/{className}/{id}",
                "openapi_field":  "put, patch",
                "openapi_value":  "PUT, PATCH",
            },
        },
        "reason": "openapi_mapping.openapi_field must be a single field, not 'put, patch'",
        "stage": "semantic",
    },
]

full_plan = planner_result["extraction_plan"]
high_priority = [s for s in full_plan["sections_to_extract"] if s["priority"] == "high"]
test_plan_sections = high_priority[:2]

# _get_sections_to_reprocess — should return only section 322
to_reprocess = _get_sections_to_reprocess(test_plan_sections, mock_errors)
assert len(to_reprocess) == 1
assert to_reprocess[0]["section_id"] == "322"
logger.info(f"_get_sections_to_reprocess OK — {len(to_reprocess)} section(s) selected.")
logger.info(f"  → [{to_reprocess[0]['section_id']}] {to_reprocess[0]['title']}")

# _build_correction_task — should include source_name, rule_type, rule_text and reason
correction = _build_correction_task(mock_errors)
assert isinstance(correction, str) and len(correction) > 0
assert "CORRECTION TASK" in correction
assert "modifyMOIAttributes" in correction
assert "put, patch" in correction
logger.info("_build_correction_task OK — correction prompt generated:")
for line in correction.splitlines():
    logger.info(f"  {line}")

# Empty on first iteration
assert _build_correction_task([]) == ""
logger.info("_build_correction_task OK — empty string on first iteration.")

# _build_allowed_keys — should return a set with one (section_id, rule_type, source_name, openapi_field) tuple
allowed = _build_allowed_keys(mock_errors)
assert isinstance(allowed, set)
assert len(allowed) == 1
expected_key = ("322", "path_operation", "modifyMOIAttributes", "put, patch")
assert expected_key in allowed
logger.info(f"_build_allowed_keys OK — {len(allowed)} key(s):")
for key in allowed:
    logger.info(f"  (section_id={key[0]}, rule_type={key[1]}, source_name={key[2]}, openapi_field={key[3]})")

In [6]:
# Step 5 — Test extractor_node() end-to-end (limited to 2 sections)
# Makes real LLM API calls. Limited to the first 2 high-priority sections
# to keep cost and latency manageable during testing.

full_plan = planner_result["extraction_plan"]
high_priority = [s for s in full_plan["sections_to_extract"] if s["priority"] == "high"]
test_plan = {**full_plan, "sections_to_extract": high_priority[:2]}

logger.info(f"Running extractor on {len(test_plan['sections_to_extract'])} section(s):")
for s in test_plan["sections_to_extract"]:
    logger.info(f"  [{s['section_id']}] {s['title']}")

extractor_state = {
    **reader_result,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "validation_errors":   [],
    "iteration_count":     0,
}

extractor_result = extractor_node(extractor_state)

assert "raw_rules" in extractor_result
assert "iteration_count" in extractor_result
assert isinstance(extractor_result["raw_rules"], list)
assert extractor_result["iteration_count"] == 1

logger.info(f"extractor_node OK")
logger.info(f"  raw_rules      : {len(extractor_result['raw_rules'])} rule(s) extracted")
logger.info(f"  iteration_count: {extractor_result['iteration_count']}")

2026-03-30 21:38:42 [INFO] __main__: Running extractor on 2 section(s):
2026-03-30 21:38:42 [INFO] __main__:   [321] 12.1.1.1 Mapping of operations
2026-03-30 21:38:42 [INFO] __main__:   [322] 12.1.1.1.1 Introduction
2026-03-30 21:38:42 [INFO] nodes.extractor: Extractor Node started.
2026-03-30 21:38:42 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-03-30 21:38:42 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3682.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect id

In [7]:
# Step 6 — Inspect raw_rules
# Reviews the extracted rules: structure, field completeness, and content.
# All rules must conform to the RawRule Pydantic schema.

raw_rules = extractor_result["raw_rules"]

# Validate each rule matches the RawRule schema
for rule_dict in raw_rules:
    RawRule(**rule_dict)  # raises ValidationError if malformed

logger.info(f"All {len(raw_rules)} rule(s) are valid RawRule objects.")
logger.info("")

for i, rule in enumerate(raw_rules):
    logger.info(f"Rule {i + 1} — [{rule['section_id']}] {rule['section_title']}")
    logger.info(f"  rule_type      : {rule['rule_type']}")
    logger.info(f"  source_name    : {rule['source_name']}")
    logger.info(f"  rule_text      : {rule['rule_text']}")
    logger.info(f"  openapi_object : {rule['openapi_mapping']['openapi_object']}")
    logger.info(f"  openapi_field  : {rule['openapi_mapping']['openapi_field']}")
    logger.info(f"  openapi_value  : {rule['openapi_mapping']['openapi_value']}")
    logger.info("")

2026-03-30 21:38:55 [INFO] __main__: All 4 rule(s) are valid RawRule objects.
2026-03-30 21:38:55 [INFO] __main__: 
2026-03-30 21:38:55 [INFO] __main__: Rule 1 — [322] 12.1.1.1.1 Introduction
2026-03-30 21:38:55 [INFO] __main__:   rule_text      : The IS operation "createMOI" maps to an OpenAPI path with HTTP method PUT at the resource URI pattern "{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}".
2026-03-30 21:38:55 [INFO] __main__:   openapi_object : paths.{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}
2026-03-30 21:38:55 [INFO] __main__:   openapi_field  : put
2026-03-30 21:38:55 [INFO] __main__:   openapi_value  : PUT
2026-03-30 21:38:55 [INFO] __main__: 
2026-03-30 21:38:55 [INFO] __main__: Rule 2 — [322] 12.1.1.1.1 Introduction
2026-03-30 21:38:55 [INFO] __main__:   rule_text      : The IS operation "getMOIAttributes" maps to an OpenAPI path with HTTP method GET at the resource URI pattern "{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-p